**Setup**

In [ ]:
!git clone https://github.com/AntonioTepsich/Convolutional-KANs.git

In [ ]:
%cd Convolutional-KANs

In [ ]:
import sys
sys.path.append('/content/Convolutional-KANs/kan_convolutional')

**Imports**

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

from kan_convolutional.KANConv import KAN_Convolutional_Layer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

**Setup for CIFAR 10 Dataset**

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [ ]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [ ]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)

In [ ]:
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
class KANC_MLP_Medium(nn.Module):
    def __init__(self, grid_size: int = 5):
        super().__init__()

        # CHANGE: CIFAR = 3 channels (not 1)
        self.conv1 = KAN_Convolutional_Layer(
            in_channels=3,
            out_channels=5,
            kernel_size=(3,3),
            grid_size=grid_size
        )

        self.conv2 = KAN_Convolutional_Layer(
            in_channels=5,
            out_channels=10,
            kernel_size=(3,3),
            grid_size=grid_size
        )

        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))

        # ADD: BatchNorm (important for CKAN stability)
        self.bn1 = nn.BatchNorm2d(5)
        self.bn2 = nn.BatchNorm2d(10)

        self.flat = nn.Flatten()

        # FIX: dynamic size (instead of hardcoded 250)
        self.linear1 = nn.LazyLinear(10)

        self.name = f"KANC MLP (Medium) (gs = {grid_size})"

    def forward(self, x):
        # ADD: activation + BN
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)

        x = self.flat(x)
        x = self.linear1(x)

        # REMOVE log_softmax
        return x

In [ ]:
# Initialization

model = KANC_MLP_Medium(grid_size=5).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
# Mixed Precision
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

In [ ]:
# Training Loop

def train(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return total_loss / len(loader), 100 * correct / total

In [ ]:
# Test Loop

def test(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / len(loader), 100 * correct / total

In [ ]:
# Training

epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader)
    test_loss, test_acc = test(model, test_loader)

    scheduler.step()

    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")